In [2]:
import pandas as pd
import numpy as np

### Importing datasets

In [67]:
quali_results = pd.read_csv("formula1-datasets-master/formula1-datasets-master/Formula1_2024season_qualifyingResults.csv")
sprint_results = pd.read_csv("formula1-datasets-master/formula1-datasets-master/Formula1_2024season_sprintResults.csv")
results_2024 = pd.read_csv("formula1-datasets-master/formula1-datasets-master/Formula1_2024season_raceResults.csv")
sprint_quali_results = pd.read_csv("formula1-datasets-master/formula1-datasets-master/Formula1_2024season_sprintQualifyingResults.csv")

In [68]:
quali_results.head()

,Track,Position,No,Driver,Team,Q1,Q2,Q3,Laps
0,Bahrain,1,1,Max Verstappen,Red Bull Racing Honda RBPT,1:30.031,1:29.374,1:29.179,17
1,Bahrain,2,16,Charles Leclerc,Ferrari,1:30.243,1:29.165,1:29.407,19
2,Bahrain,3,63,George Russell,Mercedes,1:30.350,1:29.922,1:29.485,17
3,Bahrain,4,55,Carlos Sainz,Ferrari,1:29.909,1:29.573,1:29.507,18
4,Bahrain,5,11,Sergio Perez,Red Bull Racing Honda RBPT,1:30.221,1:29.932,1:29.537,17


In [69]:
sprint_results.head() 

,Track,Position,No,Driver,Team,Starting Grid,Laps,Time/Retired,Points
0,China,1,1,Max Verstappen,Red Bull Racing Honda RBPT,4,19,32:04.660,8
1,China,2,44,Lewis Hamilton,Mercedes,2,19,+13.043,7
2,China,3,11,Sergio Perez,Red Bull Racing Honda RBPT,6,19,+15.258,6
3,China,4,16,Charles Leclerc,Ferrari,7,19,+17.486,5
4,China,5,55,Carlos Sainz,Ferrari,5,19,+20.696,4


In [70]:
results_2024.head()

,Track,Position,No,Driver,Team,Starting Grid,Laps,Time/Retired,Points,Set Fastest Lap,Fastest Lap Time
0,Bahrain,1,1,Max Verstappen,Red Bull Racing Honda RBPT,1,57,1:31:44.742,26,Yes,1:32.608
1,Bahrain,2,11,Sergio Perez,Red Bull Racing Honda RBPT,5,57,+22.457,18,No,1:34.364
2,Bahrain,3,55,Carlos Sainz,Ferrari,4,57,+25.110,15,No,1:34.507
3,Bahrain,4,16,Charles Leclerc,Ferrari,2,57,+39.669,12,No,1:34.090
4,Bahrain,5,63,George Russell,Mercedes,3,57,+46.788,10,No,1:35.065


In [71]:
sprint_quali_results.head()

,Track,Position,No,Driver,Team,Q1,Q2,Q3,Laps
0,China,1,4,Lando Norris,McLaren Mercedes,1:36.384,1:36.047,1:57.940,13
1,China,2,44,Lewis Hamilton,Mercedes,1:37.181,1:36.287,1:59.201,15
2,China,3,14,Fernando Alonso,Aston Martin Aramco Mercedes,1:36.883,1:36.119,1:59.915,14
3,China,4,1,Max Verstappen,Red Bull Racing Honda RBPT,1:36.456,1:35.606,2:00.028,12
4,China,5,55,Carlos Sainz,Ferrari,1:36.719,1:36.052,2:00.214,15


### Some Basic Info

In [72]:
results_2024.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 479 entries, 0 to 478
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Track             479 non-null    object
 1   Position          479 non-null    object
 2   No                479 non-null    int64 
 3   Driver            479 non-null    object
 4   Team              479 non-null    object
 5   Starting Grid     479 non-null    int64 
 6   Laps              479 non-null    int64 
 7   Time/Retired      479 non-null    object
 8   Points            479 non-null    int64 
 9   Set Fastest Lap   479 non-null    object
 10  Fastest Lap Time  463 non-null    object
dtypes: int64(4), object(7)
memory usage: 41.3+ KB


In [73]:
results_2024['Track'].value_counts()

Track
Bahrain           20
Saudi Arabia      20
Japan             20
China             20
Emilia Romagna    20
Miami             20
Monaco            20
Canada            20
Azerbaijan        20
Spain             20
Austria           20
Great Britain     20
Hungary           20
Belgium           20
Netherlands       20
Italy             20
Brazil            20
Singapore         20
United States     20
Mexico            20
Qatar             20
Las Vegas         20
Abu Dhabi         20
Australia         19
Name: count, dtype: int64

In [74]:
results_2024['Position'].value_counts()

Position
NC    46
1     24
3     24
4     24
5     24
2     24
6     24
7     24
9     24
8     24
11    24
12    24
13    24
10    24
14    24
15    24
16    21
17    20
18    15
19    11
20     4
DQ     2
Name: count, dtype: int64

## Initiating test to merge Kaggle driver Id with Github dataset

In [75]:
drivers_kaggle = pd.read_csv("filler-datasets/drivers_filtered.csv")

In [76]:
drivers_kaggle.head()

,driverId,driverRef,code,forename,surname
0,1,hamilton,HAM,Lewis,Hamilton
1,4,alonso,ALO,Fernando,Alonso
2,8,raikkonen,RAI,Kimi,Räikkönen
3,9,kubica,KUB,Robert,Kubica
4,20,vettel,VET,Sebastian,Vettel


### Normalizing names for matching driver's full name

In [126]:
import unicodedata

def normalize_name(s):
    if pd.isna(s):
        return s
    s = s.lower().strip()
    s = unicodedata.normalize('NFKD', s)
    s = ''.join(c for c in s if not unicodedata.combining(c))
    return s

drivers_kaggle['forename_norm'] = drivers_kaggle['forename'].apply(normalize_name)
drivers_kaggle['surname_norm']  = drivers_kaggle['surname'].apply(normalize_name)

In [84]:
drivers_kaggle=drivers_kaggle.drop('surname',axis=1)
drivers_kaggle=drivers_kaggle.drop('forename',axis=1)

In [88]:
drivers_kaggle['full_name_norm'] = drivers_kaggle['forename_norm'] + ' ' + drivers_kaggle['surname_norm']

In [90]:
drivers_kaggle=drivers_kaggle.drop('surname_norm',axis=1)
drivers_kaggle=drivers_kaggle.drop('forename_norm',axis=1)

In [91]:
drivers_kaggle.head()

,driverId,driverRef,code,full_name_norm
0,1,hamilton,HAM,lewis hamilton
1,4,alonso,ALO,fernando alonso
2,8,raikkonen,RAI,kimi raikkonen
3,9,kubica,KUB,robert kubica
4,20,vettel,VET,sebastian vettel


In [86]:
results_2024['Driver'] = results_2024['Driver'].apply(normalize_name)

#### Merging

In [96]:
results_2024 = results_2024.merge(
    drivers_kaggle[['driverId', 'full_name_norm']].drop_duplicates(),
    left_on='Driver',
    right_on='full_name_norm',
    how='left'
).rename(columns={'driverId': 'kaggle_driver_id'})


In [98]:
results_2024 = results_2024.drop(columns=['full_name_norm'])

### Checking faults which were 0

In [100]:
results_2024['kaggle_driver_id'].isna().sum()

np.int64(0)

## Test Passed so now will create pipeline to do it with all github datasets

### Function of pipeline for what we did earlier

In [134]:
def process_github_df(df):
    # normalize column names
    #df = normalize_columns(df)

    # normalize driver name
    df['full_name_norm'] = df['Driver'].apply(normalize_name)

    # map kaggle driver id
    df = df.merge(
    drivers_kaggle[['driverId', 'full_name_norm']].drop_duplicates(),
    on='full_name_norm',
    how='left'
    ).rename(columns={'driverId': 'kaggle_driver_id'})

    # drop name columns
    df = df.drop(columns=['full_name_norm'])

    return df


### Loading datasets and then storing them

In [135]:
from pathlib import Path

INPUT_DIR = Path("formula1-dataset-allresults")
OUTPUT_DIR = Path("all-dataset-processed(initial)")
OUTPUT_DIR.mkdir(exist_ok=True)

for csv_file in INPUT_DIR.glob("*.csv"):
    df = pd.read_csv(csv_file)
    df_clean = process_github_df(df)

    output_path = OUTPUT_DIR / csv_file.name
    df_clean.to_csv(output_path, index=False)

    print(f"Processed: {csv_file.name}")


Processed: formula1_2019season_raceResults.csv
Processed: formula1_2020season_raceResults.csv
Processed: formula1_2021season_raceResults.csv
Processed: formula1_2021season_sprintQualifyingResults.csv
Processed: Formula1_2022season_qualifyingResults.csv
Processed: Formula1_2022season_raceResults.csv
Processed: Formula1_2022season_sprintRaceResults.csv
Processed: Formula1_2023season_qualifyingResults.csv
Processed: Formula1_2023season_raceResults.csv
Processed: Formula1_2023season_sprintResults.csv
Processed: Formula1_2023season_sprintShootoutResults.csv
Processed: Formula1_2024season_qualifyingResults.csv
Processed: Formula1_2024season_raceResults.csv
Processed: Formula1_2024season_sprintQualifyingResults.csv
Processed: Formula1_2024season_sprintResults.csv
Processed: Formula1_2025Season_QualifyingResults.csv
Processed: Formula1_2025Season_RaceResults.csv
Processed: Formula1_2025Season_SprintQualifyingResults.csv
Processed: Formula1_2025Season_SprintResults.csv


## Working on finally making the datasets that were missing

### We need to make-
 1. 2019 and 2020 Race Qualifying
 2. 2021 Race Qualifying and Sprint Results

In [4]:
kaggle_sprint_res = pd.read_csv('filler-datasets/sprint_results.csv')
kaggle_sprint_res.head()

,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,fastestLapTime,statusId
0,1,1061,830,9,33,2,1,1,1,3,17,25:38.426,1538426,14,1:30.013,1
1,2,1061,1,131,44,1,2,2,2,2,17,+1.430,1539856,17,1:29.937,1
2,3,1061,822,131,77,3,3,3,3,1,17,+7.502,1545928,17,1:29.958,1
3,4,1061,844,6,16,4,4,4,4,0,17,+11.278,1549704,16,1:30.163,1
4,5,1061,846,1,4,6,5,5,5,0,17,+24.111,1562537,16,1:30.566,1


In [5]:
kaggle_race = pd.read_csv('filler-datasets/races_filtered.csv')
kaggle_race.head()

,raceId,year,round,circuitId,name
0,989,2018,1,1,Australian Grand Prix
1,990,2018,2,3,Bahrain Grand Prix
2,991,2018,3,17,Chinese Grand Prix
3,992,2018,4,73,Azerbaijan Grand Prix
4,993,2018,5,4,Spanish Grand Prix


In [8]:
len(kaggle_sprint_res['raceId'].value_counts())

18

In [10]:
len(kaggle_race['raceId'].value_counts())

149

### Adding year to kaggle_sprint_result

In [11]:
kaggle_sprint_res_check = kaggle_sprint_res.merge(
    kaggle_race[['raceId', 'year']],
    on='raceId',
    how='left'
)

In [15]:
kaggle_sprint_res_check['year'].value_counts()
#kaggle_sprint_res_check['year'].isna().sum()

year
2024    120
2023    120
2022     60
2021     60
Name: count, dtype: int64

In [17]:
kaggle_sprint_res = kaggle_sprint_res_check
kaggle_sprint_res.head()

,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,fastestLapTime,statusId,year
0,1,1061,830,9,33,2,1,1,1,3,17,25:38.426,1538426,14,1:30.013,1,2021
1,2,1061,1,131,44,1,2,2,2,2,17,+1.430,1539856,17,1:29.937,1,2021
2,3,1061,822,131,77,3,3,3,3,1,17,+7.502,1545928,17,1:29.958,1,2021
3,4,1061,844,6,16,4,4,4,4,0,17,+11.278,1549704,16,1:30.163,1,2021
4,5,1061,846,1,4,6,5,5,5,0,17,+24.111,1562537,16,1:30.566,1,2021


In [20]:
# Pass a LIST of column names
kaggle_sprint_res = kaggle_sprint_res.drop(['position', 'positionText'], axis=1)

In [21]:
kaggle_sprint_res.head()

,resultId,raceId,driverId,constructorId,number,grid,positionOrder,points,laps,time,milliseconds,fastestLap,fastestLapTime,statusId,year
0,1,1061,830,9,33,2,1,3,17,25:38.426,1538426,14,1:30.013,1,2021
1,2,1061,1,131,44,1,2,2,17,+1.430,1539856,17,1:29.937,1,2021
2,3,1061,822,131,77,3,3,1,17,+7.502,1545928,17,1:29.958,1,2021
3,4,1061,844,6,16,4,4,0,17,+11.278,1549704,16,1:30.163,1,2021
4,5,1061,846,1,4,6,5,0,17,+24.111,1562537,16,1:30.566,1,2021


In [23]:
git_sprint_res = pd.read_csv('all-dataset-processed(initial)/Formula1_2024season_sprintResults.csv')
git_sprint_res.head()

,Track,Position,No,Driver,Team,Starting Grid,Laps,Time/Retired,Points,kaggle_driver_id
0,China,1,1,Max Verstappen,Red Bull Racing Honda RBPT,4,19,32:04.660,8,830
1,China,2,44,Lewis Hamilton,Mercedes,2,19,+13.043,7,1
2,China,3,11,Sergio Perez,Red Bull Racing Honda RBPT,6,19,+15.258,6,815
3,China,4,16,Charles Leclerc,Ferrari,7,19,+17.486,5,844
4,China,5,55,Carlos Sainz,Ferrari,5,19,+20.696,4,832


## Realising that we first need to finalize the features we will be using so that we can avoid future issues

### For - 
    1.Race
        season
        round
        kaggle_driver_id
        Position
        Starting Grid
        Laps
        Time/Retired
        milliseconds
        has_fastest_lap 

    2.Quali
        season
        round
        kaggle_driver_id
        Position (quali position)
        Q1
        Q2
        Q3

    3.Sprint_Race
        season
        round
        kaggle_driver_id
        Position
        Starting Grid
        Laps
        Time/Retired
        milliseconds

    4.Sprint_Race_Quali
        season
        round
        kaggle_driver_id
        Position (quali position)
        Q1
        Q2
        Q3

In [89]:
race_2019 = pd.read_csv('all-dataset-processed(initial)/formula1_2019season_raceResults.csv')
race_2020 = pd.read_csv('all-dataset-processed(initial)/formula1_2020season_raceResults.csv')
race_2021 = pd.read_csv('all-dataset-processed(initial)/formula1_2021season_raceResults.csv')
race_2022 = pd.read_csv('all-dataset-processed(initial)/formula1_2022season_raceResults.csv')
race_2023 = pd.read_csv('all-dataset-processed(initial)/formula1_2023season_raceResults.csv')
race_2024 = pd.read_csv('all-dataset-processed(initial)/formula1_2024season_raceResults.csv')
race_2025 = pd.read_csv('all-dataset-processed(initial)/formula1_2025season_raceResults.csv')

 ### Removing 2018 data as github 2018 data not available

In [52]:
kaggle_race = pd.read_csv('filler-datasets/races_filtered.csv')
kaggle_race = kaggle_race[kaggle_race['year']>2018]
kaggle_race.to_csv('filler-datasets/kaggle_race.csv', index=False)

 ## Since we are to make a common(final) dataset but we do not have any means to merge the milliseconds as we cannot use Race Name as it is different and not by country too as 2 races can be in the same country (Miami GP and US GP) so we check if any of the races are missing and if not we assume that all races are present and in order so we can provide an ID so it will be easy to add data in future too.

## Id merging

In [47]:
len(kaggle_race[kaggle_race['year']>2018])

128

In [51]:
len(race_2019['Track'].value_counts()) + len(race_2020['Track'].value_counts()) + len(race_2021['Track'].value_counts()) + len(race_2022['Track'].value_counts()) + len(race_2023['Track'].value_counts()) + len(race_2024['Track'].value_counts())

128

In [53]:
race_2019.head()

,Track,Position,No,Driver,Team,Starting Grid,Laps,Total Time/Gap/Retirement,Points,Fastest Lap,kaggle_driver_id
0,Australia,1,77,Valtteri Bottas,Mercedes,2,58,1:25:27.325,26,Yes,822
1,Australia,2,44,Lewis Hamilton,Mercedes,1,58,+20.886,18,No,1
2,Australia,3,33,Max Verstappen,Red Bull Racing Honda,4,58,+22.520,15,No,830
3,Australia,4,5,Sebastian Vettel,Ferrari,3,58,+57.109,12,No,20
4,Australia,5,16,Charles Leclerc,Ferrari,5,58,+58.230,10,No,844


In [54]:
kaggle_race.head()

,raceId,year,round,circuitId,name
21,1010,2019,1,1,Australian Grand Prix
22,1011,2019,2,3,Bahrain Grand Prix
23,1012,2019,3,17,Chinese Grand Prix
24,1013,2019,4,73,Azerbaijan Grand Prix
25,1014,2019,5,4,Spanish Grand Prix


### Checking if years and rounds is monotonically increasing in Kaggle Dataset so that giving raceId to GitHub dataset doesn't have issue

In [ ]:
bad_years = (
    kaggle_race
    .groupby('year')['round']
    .apply(lambda x: not x.is_monotonic_increasing)
)

bad_years[bad_years]

year = bad_years[bad_years].index[0]

kaggle_race[kaggle_race['year'] == year][['raceId', 'round', 'name']]


### ↑ Year 2021 was found not monotonically not increasing so sorted it. (Result of above snippet not seen as accidently ran code again after sorting)

### Sorting it 

In [66]:
kaggle_race = (
    kaggle_race
    .sort_values(['year', 'round', 'raceId'])
    .reset_index(drop=True)
)
assert kaggle_race.groupby('year')['round'].apply(
    lambda x: x.is_monotonic_increasing
).all()
## since it returns nothing that means it is monotonically increasing

In [67]:
kaggle_race.to_csv('filler-datasets/kaggle_race.csv', index=False)

### Created a function which assigned raceId on the logic that when position 1 was found that means new race has started 

In [10]:
def assign_race_ids(
    race_dfs,        # list: [race_2019, ..., race_2024]
    kaggle_race,     # dataframe with raceId, year, round
    drivers_per_race # int, e.g. 20
):
    kaggle_idx = 0
    output_dfs = []

    for race_df in race_dfs:
        race_df = race_df.copy()
        race_ids = []

        i = 0
        while i < len(race_df):
            if kaggle_idx >= len(kaggle_race):
                raise ValueError("❌ Kaggle race index exceeded — mismatch detected")

            race_id = kaggle_race.iloc[kaggle_idx]['raceId']

            race_ids.extend([race_id] * drivers_per_race)

            i += drivers_per_race
            kaggle_idx += 1

        if len(race_ids) != len(race_df):
            raise ValueError("❌ Driver count mismatch while assigning raceId")

        race_df['raceId'] = race_ids
        output_dfs.append(race_df)

    if kaggle_idx != len(kaggle_race):
        raise ValueError("❌ Kaggle races left unassigned — mismatch detected")

    print("✅ raceId assignment successful")
    return output_dfs


In [11]:
race_dfs = [
    race_2019,
    race_2020,
    race_2021,
    race_2022,
    race_2023,
    race_2024
]

race_2019_check, race_2020_check, race_2021_check, race_2022_check, race_2023_check, race_2024_check = \
    assign_race_ids(race_dfs, kaggle_race, drivers_per_race=20)


ValueError: ❌ Driver count mismatch while assigning raceId

### Driver mismatch so we check why this error is happening

In [12]:
for year, df in zip(
    [2019, 2020, 2021, 2022, 2023, 2024],
    race_dfs
):
    print(year, len(df), "rows", "mod 20 =", len(df) % 20)


2019 420 rows mod 20 = 0
2020 340 rows mod 20 = 0
2021 440 rows mod 20 = 0
2022 440 rows mod 20 = 0
2023 440 rows mod 20 = 0
2024 479 rows mod 20 = 19


### Apart from 2024 all the races have complete entries (2024 is missing one entry)

In [37]:
race_2024['Position'] = race_2024['Position'].astype(str)

race_starts = race_2024.index[race_2024['Position'] == '1'].tolist()
race_starts.append(len(race_2024))


### We created a function (it was deleted earlier) which retured us -> (2, 19), (12, 21), (13, 19), that means 3 blocks did not have the matching 20 drivers per race. So we analyzed those indexes
def find_bad_race_blocks(df, drivers_per_race=20):
    """
    Returns a list of (race_index, block_size)
    for races that do not have `drivers_per_race` rows.
    """
    bad_blocks = []
    race_idx = 0
    i = 0

    while i < len(df):
        block = df.iloc[i:i + drivers_per_race]
        if len(block) != drivers_per_race:
            bad_blocks.append((race_idx, len(block)))
            break  # last incomplete block
        i += drivers_per_race
        race_idx += 1

    return bad_blocks


### Only 19 drivers in Australian GP

In [9]:
i = 2
start = race_starts[i]
end = race_starts[i+1]

race_2024.iloc[start:end][
    ['Track', 'Position', 'No', 'Driver', 'kaggle_driver_id']
]


,Track,Position,No,Driver,kaggle_driver_id
40,Australia,1,55,Carlos Sainz,832
41,Australia,2,16,Charles Leclerc,844
42,Australia,3,4,Lando Norris,846
43,Australia,4,81,Oscar Piastri,857
44,Australia,5,11,Sergio Perez,815
45,Australia,6,18,Lance Stroll,840
46,Australia,7,22,Yuki Tsunoda,852
47,Australia,8,14,Fernando Alonso,4
48,Australia,9,27,Nico Hulkenberg,807
49,Australia,10,20,Kevin Magnussen,825


### Due to Goerge Russel DQ the logic of assigning a *Race* broke so we will have to change the logic

In [10]:
i = 12
start = race_starts[i]
end = race_starts[i+1]

race_2024.iloc[start:end][
    ['Track', 'Position', 'No', 'Driver', 'kaggle_driver_id']
]


,Track,Position,No,Driver,kaggle_driver_id
239,Hungary,1,81,Oscar Piastri,857
240,Hungary,2,4,Lando Norris,846
241,Hungary,3,44,Lewis Hamilton,1
242,Hungary,4,16,Charles Leclerc,844
243,Hungary,5,1,Max Verstappen,830
244,Hungary,6,55,Carlos Sainz,832
245,Hungary,7,11,Sergio Perez,815
246,Hungary,8,63,George Russell,847
247,Hungary,9,22,Yuki Tsunoda,852
248,Hungary,10,18,Lance Stroll,840


In [11]:
i = 13
start = race_starts[i]
end = race_starts[i+1]

race_2024.iloc[start:end][
    ['Track', 'Position', 'No', 'Driver', 'kaggle_driver_id']
]


,Track,Position,No,Driver,kaggle_driver_id
260,Belgium,1,44,Lewis Hamilton,1
261,Belgium,2,81,Oscar Piastri,857
262,Belgium,3,16,Charles Leclerc,844
263,Belgium,4,1,Max Verstappen,830
264,Belgium,5,4,Lando Norris,846
265,Belgium,6,55,Carlos Sainz,832
266,Belgium,7,11,Sergio Perez,815
267,Belgium,8,14,Fernando Alonso,4
268,Belgium,9,31,Esteban Ocon,839
269,Belgium,10,3,Daniel Ricciardo,817


In [22]:
race_2024.iloc[258:261]

,Track,Position,No,Driver,Team,Starting Grid,Laps,Time/Retired,Points,Set Fastest Lap,Fastest Lap Time,kaggle_driver_id
258,Hungary,NC,10,Pierre Gasly,Alpine Renault,20,33,DNF,0,No,1:23.340,842
259,Belgium,DQ,63,George Russell,Mercedes,6,44,1:19:57.040,0,No,1:47.113,847
260,Belgium,1,44,Lewis Hamilton,Mercedes,3,44,+0.526,25,No,1:46.653,1


In [21]:
kag_race = pd.read_csv('filler-datasets/results.csv')
kag_race.head()
#kag_race['grid'].value_counts()

,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,rank,fastestLapTime,fastestLapSpeed,statusId
0,1,18,1,1,22,1,1,1,1,10.0,58,1:34:50.616,5690616,39,2,1:27.452,218.300,1
1,2,18,2,2,3,5,2,2,2,8.0,58,+5.478,5696094,41,3,1:27.739,217.586,1
2,3,18,3,3,7,7,3,3,3,6.0,58,+8.163,5698779,41,5,1:28.090,216.719,1
3,4,18,4,4,5,11,4,4,4,5.0,58,+17.181,5707797,58,7,1:28.603,215.464,1
4,5,18,5,1,23,3,5,5,5,4.0,58,+18.014,5708630,43,1,1:27.418,218.385,1


### We found that Logan Sargeant's entry was missing from Austrailian GP as he was marked DNS so we found his details and added him to out dataset

In [25]:
drivers = pd.read_csv('filler-datasets/drivers_filtered.csv')
drivers[drivers['forename'] =='Logan']

,driverId,driverRef,code,forename,surname
35,858,sargeant,SAR,Logan,Sargeant


In [27]:
driver = pd.read_csv('formula1-datasets-master/formula1-datasets-master/Formula1_2024season_drivers.csv')
driver[driver['Abbreviation']== 'SAR']

,Driver,Abbreviation,No,Team,Country,Podiums,Points,Grands Prix Entered,World Championships,Highest Race Finish,Highest Grid Position,Date of Birth,Place of Birth
22,Logan Sargeant,SAR,2,Williams,United States,0,1.0,36,0,10(x1),6,31/12/2000,"Fort Lauderdale, Florida"


In [28]:
new_row = {
    'Track': 'Australia',
    'Position': 'NC',
    'No': 2,
    'Driver': 'Logan Sargeant',
    'kaggle_driver_id': 858
}

# Split + insert
race_2024 = pd.concat(
    [
        race_2024.iloc[:59],
        pd.DataFrame([new_row]),
        race_2024.iloc[59:]
    ],
    ignore_index=True
)

### New logic that labels the new race as the "Track" column changes

In [30]:
def assign_race_ids(
    race_dfs,        # list: [race_2019, ..., race_2024]
    kaggle_race,     # dataframe with raceId, year, round
    drivers_per_race # int, e.g. 20
):
    kaggle_idx = 0
    output_dfs = []

    for race_df in race_dfs:
        race_df = race_df.copy()
        race_ids = []

        i = 0
        while i < len(race_df):
            if kaggle_idx >= len(kaggle_race):
                raise ValueError("❌ Kaggle race index exceeded — mismatch detected")

            race_id = kaggle_race.iloc[kaggle_idx]['raceId']

            race_ids.extend([race_id] * drivers_per_race)

            i += drivers_per_race
            kaggle_idx += 1

        if len(race_ids) != len(race_df):
            raise ValueError("❌ Driver count mismatch while assigning raceId")

        race_df['raceId'] = race_ids
        output_dfs.append(race_df)

    if kaggle_idx != len(kaggle_race):
        raise ValueError("❌ Kaggle races left unassigned — mismatch detected")

    print("✅ raceId assignment successful")
    return output_dfs


In [31]:
race_dfs = [
    race_2019,
    race_2020,
    race_2021,
    race_2022,
    race_2023,
    race_2024
]

race_2019_check, race_2020_check, race_2021_check, race_2022_check, race_2023_check, race_2024_check = \
    assign_race_ids(race_dfs, kaggle_race, drivers_per_race=20)


✅ raceId assignment successful


In [35]:
race_2019_check.head()

,Track,Position,No,Driver,Team,Starting Grid,Laps,Total Time/Gap/Retirement,Points,Fastest Lap,kaggle_driver_id,raceId
0,Australia,1,77,Valtteri Bottas,Mercedes,2,58,1:25:27.325,26,Yes,822,1010
1,Australia,2,44,Lewis Hamilton,Mercedes,1,58,+20.886,18,No,1,1010
2,Australia,3,33,Max Verstappen,Red Bull Racing Honda,4,58,+22.520,15,No,830,1010
3,Australia,4,5,Sebastian Vettel,Ferrari,3,58,+57.109,12,No,20,1010
4,Australia,5,16,Charles Leclerc,Ferrari,5,58,+58.230,10,No,844,1010


In [36]:
race_2019_check.to_csv('all-dataset-processed(initial)/formula1_2019season_raceResults.csv', index=False)
race_2020_check.to_csv('all-dataset-processed(initial)/formula1_2020season_raceResults.csv', index=False)
race_2021_check.to_csv('all-dataset-processed(initial)/formula1_2021season_raceResults.csv', index=False)
race_2022_check.to_csv('all-dataset-processed(initial)/formula1_2022season_raceResults.csv', index=False)
race_2023_check.to_csv('all-dataset-processed(initial)/formula1_2023season_raceResults.csv', index=False)
race_2024_check.to_csv('all-dataset-processed(initial)/formula1_2024season_raceResults.csv', index=False)

## Now dealing with the Time column

### It has either the exact time or "+x" seconds or "+y" laps

In [71]:
race_2019['Total Time/Gap/Retirement'].value_counts()

Total Time/Gap/Retirement
+1 lap     135
DNF         58
+2 laps     27
+3 laps      4
+57.109      1
          ... 
+16.772      1
+43.435      1
+44.379      1
+64.357      1
+69.205      1
Name: count, Length: 200, dtype: int64

In [75]:
kaggle_race = pd.read_csv('filler-datasets/results_filtered.csv')
kaggle_race.head()

,resultId,raceId,driverId,position,time,milliseconds,fastestLapTime,fastestLapSpeed
0,23782,989,20,1,1:29:33.283,5373283,1:26.469,220.782
1,23783,989,1,2,+5.036,5378319,1:26.444,220.845
2,23784,989,8,3,+6.309,5379592,1:26.373,221.027
3,23785,989,817,4,+7.069,5380352,1:25.945,222.128
4,23786,989,4,5,+27.886,5401169,1:26.978,219.489


### Transforming the column into a structure where if dnf/dsq then there is a separate column for that and if the driver is laps down then there is column for that which shows how many laps down is the driver and for time the race winner has 0 assigned and the rest driver time with "+x" seconds will have x as their data

In [90]:
def parse_time_gap_retired(val):
    if pd.isna(val):
        return pd.Series([0, 1, np.nan, np.nan])

    v = str(val).strip().lower()

    # Explicit non-finish
    if v in ['dnf', 'nc', 'dns', 'dsq', 'retired']:
        return pd.Series([0, 1, np.nan, np.nan])

    # Winner
    if ':' in v:
        return pd.Series([1, 0, 0, 0.0])

    # Finished but lapped
    if 'lap' in v:
        laps = int(re.search(r'\d+', v).group())
        return pd.Series([1, 0, laps, np.nan])

    # Finished with time gap
    if v.startswith('+'):
        return pd.Series([1, 0, 0, float(v[1:])])

    # Safety fallback (should almost never happen)
    return pd.Series([0, 1, np.nan, np.nan])


### Checking with 2019 data

In [91]:
race_2019[
    ['finished', 'dnf', 'laps_down', 'time_gap_sec']
] = race_2019['Total Time/Gap/Retirement'].apply(parse_time_gap_retired)


In [92]:
# DNF sanity
race_2019[['dnf', 'finished']].value_counts()

dnf  finished
0.0  1.0         362
1.0  0.0          58
Name: count, dtype: int64

In [93]:
# Laps down sanity
race_2019['laps_down'].value_counts(dropna=False)

laps_down
0.0    196
1.0    135
NaN     58
2.0     27
3.0      4
Name: count, dtype: int64

In [94]:
# Time gap sanity
race_2019['time_gap_sec'].describe()

count    196.000000
mean      34.330260
std       29.648372
min        0.000000
25%        8.148500
50%       27.024500
75%       58.913000
max      109.838000
Name: time_gap_sec, dtype: float64

In [95]:
# Spot check DNFs
race_2019[race_2019['dnf'] == 1][
    ['Position', 'Total Time/Gap/Retirement']
].head()

,Position,Total Time/Gap/Retirement
17,NC,DNF
18,NC,DNF
19,NC,DNF
36,17,DNF
37,18,DNF


In [109]:
## 2019 and 2020 already processed but cell deleted (their column name were different)

### Processing the rest of the data

In [108]:
race_datasets = {
    2021: race_2021,
    2022: race_2022,
    2023: race_2023,
    2024: race_2024,
}
for year, df in race_datasets.items():
    df[['finished', 'dnf', 'laps_down', 'time_gap_sec']] = \
        df['Time/Retired'].apply(parse_time_gap_retired)

    df.drop(columns=['Time/Retired'], inplace=True)

    print(f"✅ {year} processed")

✅ 2021 processed
✅ 2022 processed
✅ 2023 processed
✅ 2024 processed


In [122]:
race_datasets = {
    2025: race_2025,
}
for year, df in race_datasets.items():
    df[['finished', 'dnf', 'laps_down', 'time_gap_sec']] = \
        df['Time/Retired'].apply(parse_time_gap_retired)

    df.drop(columns=['Time/Retired'], inplace=True)

    print(f"✅ {year} processed")

✅ 2025 processed


In [110]:
race_2021.head()

,Track,Position,No,Driver,Team,Starting Grid,Laps,Points,+1 Pt,Fastest Lap,kaggle_driver_id,raceId,finished,dnf,laps_down,time_gap_sec
0,Bahrain,1,44,Lewis Hamilton,Mercedes,2.0,56,25.0,No,1:34.015,1,1052,1.0,0.0,0.0,0.000
1,Bahrain,2,33,Max Verstappen,Red Bull Racing Honda,1.0,56,18.0,No,1:33.228,830,1052,1.0,0.0,0.0,0.745
2,Bahrain,3,77,Valtteri Bottas,Mercedes,3.0,56,16.0,Yes,1:32.090,822,1052,1.0,0.0,0.0,37.383
3,Bahrain,4,4,Lando Norris,McLaren Mercedes,7.0,56,12.0,No,1:34.396,846,1052,1.0,0.0,0.0,46.466
4,Bahrain,5,11,Sergio Perez,Red Bull Racing Honda,11.0,56,10.0,No,1:33.970,815,1052,1.0,0.0,0.0,52.047


In [113]:
for year, df in race_datasets.items():
    assert (df['finished'] + df['dnf'] == 1).all()
    print(year, "OK")


2021 OK
2022 OK
2023 OK
2024 OK


### Finalizing the columns of Race Dataset

In [114]:
def clean_race_columns(df):
    """
    Keeps only the final approved race-result columns.
    Points are explicitly removed.
    """

    cols_to_keep = [
        'Track',
        'Team',
        'raceId',
        'kaggle_driver_id',
        'Position',
        'Starting Grid',
        'finished',
        'dnf',
        'laps_down',
        'time_gap_sec'
    ]

    df.drop(
        columns=[c for c in df.columns if c not in cols_to_keep],
        inplace=True
    )

    return df
race_datasets = {
    2019: race_2019,
    2020: race_2020,
    2021: race_2021,
    2022: race_2022,
    2023: race_2023,
    2024: race_2024,
}

for year, df in race_datasets.items():
    race_datasets[year] = clean_race_columns(df)
    print(f"✅ {year} cleaned")


✅ 2019 cleaned
✅ 2020 cleaned
✅ 2021 cleaned
✅ 2022 cleaned
✅ 2023 cleaned
✅ 2024 cleaned


In [124]:
race_datasets = {
    2025: race_2025,
}

for year, df in race_datasets.items():
    race_datasets[year] = clean_race_columns(df)
    print(f"✅ {year} cleaned")

✅ 2025 cleaned


In [246]:
race_2019.to_csv('formula1_2019season_raceResults.csv', index=False)
race_2020.to_csv('formula1_2020season_raceResults.csv', index=False)
race_2021.to_csv('formula1_2021season_raceResults.csv', index=False)
race_2022.to_csv('formula1_2022season_raceResults.csv', index=False)
race_2023.to_csv('formula1_2023season_raceResults.csv', index=False)
race_2024.to_csv('formula1_2024season_raceResults.csv', index=False)

## Minor issue

### Wanted to standardize the 2025 dataset too where I noticed some kaggle_driver_id to be NaN

In [125]:
race_2025.head()

,Position,Starting Grid,kaggle_driver_id,finished,dnf,laps_down,time_gap_sec
0,1,1,846.0,1.0,0.0,0.0,0.000
1,2,3,830.0,1.0,0.0,0.0,0.895
2,3,4,847.0,1.0,0.0,0.0,8.481
3,4,16,NaN,1.0,0.0,0.0,10.135
4,5,6,848.0,1.0,0.0,0.0,12.773


### Found that 3 drivers did not have kaggle_driver_id (as 72 = 24*3)

In [130]:
race_2025[race_2025['kaggle_driver_id'].isna()]

,Position,Starting Grid,kaggle_driver_id,finished,dnf,laps_down,time_gap_sec
3,4,16,NaN,1.0,0.0,0.0,10.135
15,NC,15,NaN,0.0,1.0,NaN,NaN
19,NC,11,NaN,0.0,1.0,NaN,NaN
25,6,8,NaN,1.0,0.0,0.0,53.748
30,11,7,NaN,1.0,0.0,0.0,78.875
...,...,...,...,...,...,...,...
451,13,19,NaN,1.0,0.0,0.0,84.812
456,18,6,NaN,0.0,1.0,NaN,NaN
469,11,7,NaN,1.0,0.0,0.0,81.043
473,15,14,NaN,1.0,0.0,0.0,84.399


In [134]:
kaggle_driver = pd.read_csv('filler-datasets/drivers.csv')
kaggle_driver.tail(5)

,driverId,driverRef,number,code,forename,surname,dob,nationality,url
856,858,sargeant,2,SAR,Logan,Sargeant,2000-12-31,American,http://en.wikipedia.org/wiki/Logan_Sargeant
857,859,lawson,30,LAW,Liam,Lawson,2002-02-11,New Zealander,http://en.wikipedia.org/wiki/Liam_Lawson
858,860,bearman,38,BEA,Oliver,Bearman,2005-05-08,British,http://en.wikipedia.org/wiki/Oliver_Bearman
859,861,colapinto,43,COL,Franco,Colapinto,2003-05-27,Argentinian,http://en.wikipedia.org/wiki/Franco_Colapinto
860,862,doohan,61,DOO,Jack,Doohan,2003-01-20,Australian,http://en.wikipedia.org/wiki/Jack_Doohan


In [135]:
race_2025_ori = pd.read_csv('all-dataset-processed(initial)/Formula1_2025Season_RaceResults.csv')

### 3 drivers that were not there in dataset were found using checking thier position with the internet and now will be giving the kaggle_driver_id to them after checking the last kaggle_driver_id to avoid id conflicts

In [138]:
race_2025_ori['driver_norm'] = (
    race_2025_ori['Driver']
    .str.strip()
    .str.lower()
)
temp_driver_map = {
    'kimi antonelli': 863,
    'isack hadjar': 864,
    'gabriel bortoleto': 865
}
mask = race_2025_ori['kaggle_driver_id'].isna()

race_2025_ori.loc[mask, 'kaggle_driver_id'] = (
    race_2025_ori.loc[mask, 'driver_norm']
    .map(temp_driver_map)
)
# No NaNs left
#assert race_2025_ori['kaggle_driver_id'].isna().sum() == 0

# Check counts
race_2025_ori['kaggle_driver_id'].value_counts().loc[[863, 864, 865]]
race_2025_ori.drop(columns=['driver_norm'], inplace=True)

In [139]:
race_2025_ori.head()

,Track,Position,No,Driver,Team,Starting Grid,Laps,Time/Retired,Points,Set Fastest Lap,Fastest Lap Time,kaggle_driver_id
0,Australia,1,4,Lando Norris,McLaren Mercedes,1,57,1:42:06.304,25,Yes,1:22.167,846.0
1,Australia,2,1,Max Verstappen,Red Bull Racing Honda RBPT,3,57,+0.895,18,No,1:23.081,830.0
2,Australia,3,63,George Russell,Mercedes,4,57,+8.481,15,No,1:25.065,847.0
3,Australia,4,12,Kimi Antonelli,Mercedes,16,57,+10.135,12,No,1:24.901,863.0
4,Australia,5,23,Alexander Albon,Williams Mercedes,6,57,+12.773,10,No,1:24.597,848.0


### Now reprocessing the *Correct* 2025 race result dataset

In [140]:
race_datasets = {
    2025: race_2025_ori,
}
for year, df in race_datasets.items():
    df[['finished', 'dnf', 'laps_down', 'time_gap_sec']] = \
        df['Time/Retired'].apply(parse_time_gap_retired)

    df.drop(columns=['Time/Retired'], inplace=True)

    print(f"✅ {year} processed")
    
race_datasets = {
    2025: race_2025_ori,
}

for year, df in race_datasets.items():
    race_datasets[year] = clean_race_columns(df)
    print(f"✅ {year} cleaned")

✅ 2025 processed
✅ 2025 cleaned


In [141]:
race_2025_ori.head()

,Position,Starting Grid,kaggle_driver_id,finished,dnf,laps_down,time_gap_sec
0,1,1,846.0,1.0,0.0,0.0,0.000
1,2,3,830.0,1.0,0.0,0.0,0.895
2,3,4,847.0,1.0,0.0,0.0,8.481
3,4,16,863.0,1.0,0.0,0.0,10.135
4,5,6,848.0,1.0,0.0,0.0,12.773


In [247]:
race_2025_ori.to_csv('formula1_2025Season_RaceResults.csv', index=False)

### Adding the new driver's info to kaggle_driver dataset

In [145]:
new_drivers = pd.DataFrame([
    {
        'driverId': 863,
        'driverRef': 'antonelli',
        'number': None,
        'code': 'ANT',
        'forename': 'Kimi',
        'surname': 'Antonelli',
        'dob': None,
        'nationality': 'Italian',
        'url': 'https://en.wikipedia.org/wiki/Kimi_Antonelli'
    },
    {
        'driverId': 864,
        'driverRef': 'hadjar',
        'number': None,
        'code': 'HAD',
        'forename': 'Isack',
        'surname': 'Hadjar',
        'dob': None,
        'nationality': 'French',
        'url': 'https://en.wikipedia.org/wiki/Isack_Hadjar'
    },
    {
        'driverId': 865,
        'driverRef': 'bortoleto',
        'number': None,
        'code': 'BOR',
        'forename': 'Gabriel',
        'surname': 'Bortoleto',
        'dob': None,
        'nationality': 'Brazilian',
        'url': 'https://en.wikipedia.org/wiki/Gabriel_Bortoleto'
    }
])

kaggle_driver = pd.concat(
    [kaggle_driver, new_drivers],
    ignore_index=True
)

# Check they exist
kaggle_driver[kaggle_driver['driverId'].isin([863, 864, 865])]

# Ensure no duplicate driverIds
#assert kaggle_driver['driverId'].is_unique


,driverId,driverRef,number,code,forename,surname,dob,nationality,url
861,863,antonelli,None,ANT,Kimi,Antonelli,None,Italian,https://en.wikipedia.org/wiki/Kimi_Antonelli
862,864,hadjar,None,HAD,Isack,Hadjar,None,French,https://en.wikipedia.org/wiki/Isack_Hadjar
863,865,bortoleto,None,BOR,Gabriel,Bortoleto,None,Brazilian,https://en.wikipedia.org/wiki/Gabriel_Bortoleto
864,863,antonelli,None,ANT,Kimi,Antonelli,None,Italian,https://en.wikipedia.org/wiki/Kimi_Antonelli
865,864,hadjar,None,HAD,Isack,Hadjar,None,French,https://en.wikipedia.org/wiki/Isack_Hadjar
866,865,bortoleto,None,BOR,Gabriel,Bortoleto,None,Brazilian,https://en.wikipedia.org/wiki/Gabriel_Bortoleto


In [149]:
kaggle_driver.to_csv('filler-datasets/drivers_filtered.csv', index=False)

## Working on the Sprint Result dataset

In [164]:
kaggle_sprint = pd.read_csv('filler-datasets/sprint_results.csv')
sprint_2022 = pd.read_csv('all-dataset-processed(initial)/Formula1_2022season_sprintRaceResults.csv')
sprint_2023 = pd.read_csv('all-dataset-processed(initial)/Formula1_2023season_sprintResults.csv')
sprint_2024 = pd.read_csv('all-dataset-processed(initial)/Formula1_2024season_sprintResults.csv')
sprint_2025 = pd.read_csv('all-dataset-processed(initial)/Formula1_2025Season_SprintResults.csv')

In [165]:
sprint_2022.head()

,Track,Position,No,Driver,Team,Starting Grid,Laps,Time/Retired,Points,kaggle_driver_id
0,Emilia Romagna,1,1,Max Verstappen,Red Bull Racing RBPT,1,21,30:39.567,8,830
1,Emilia Romagna,2,16,Charles Leclerc,Ferrari,2,21,+2.975,7,844
2,Emilia Romagna,3,11,Sergio Perez,Red Bull Racing RBPT,7,21,+4.721,6,815
3,Emilia Romagna,4,55,Carlos Sainz,Ferrari,10,21,+17.578,5,832
4,Emilia Romagna,5,4,Lando Norris,McLaren Mercedes,3,21,+24.561,4,846


In [166]:
sprint_2024.head()

,Track,Position,No,Driver,Team,Starting Grid,Laps,Time/Retired,Points,kaggle_driver_id
0,China,1,1,Max Verstappen,Red Bull Racing Honda RBPT,4,19,32:04.660,8,830
1,China,2,44,Lewis Hamilton,Mercedes,2,19,+13.043,7,1
2,China,3,11,Sergio Perez,Red Bull Racing Honda RBPT,6,19,+15.258,6,815
3,China,4,16,Charles Leclerc,Ferrari,7,19,+17.486,5,844
4,China,5,55,Carlos Sainz,Ferrari,5,19,+20.696,4,832


In [167]:
sprint_2025['driver_norm'] = (
    sprint_2025['Driver']
    .str.strip()
    .str.lower()
)
temp_driver_map = {
    'kimi antonelli': 863,
    'isack hadjar': 864,
    'gabriel bortoleto': 865
}
mask = sprint_2025['kaggle_driver_id'].isna()

sprint_2025.loc[mask, 'kaggle_driver_id'] = (
    sprint_2025.loc[mask, 'driver_norm']
    .map(temp_driver_map)
)
sprint_2025.drop(columns=['driver_norm'], inplace=True)

In [168]:
sprint_2025[sprint_2025['kaggle_driver_id'].isna()]

,Track,Position,No,Driver,Team,Starting Grid,Laps,Time/Retired,Points,kaggle_driver_id


In [189]:
# kaggle_sprint data has data of 18 races (year 2021-2024)
len(kaggle_sprint[kaggle_sprint['position']=='1'])

18

In [190]:
kaggle_sprint['raceId'].value_counts()

raceId
1061    20
1065    20
1071    20
1077    20
1084    20
1095    20
1101    20
1107    20
1110    20
1115    20
1116    20
1118    20
1125    20
1126    20
1131    20
1139    20
1141    20
1143    20
Name: count, dtype: int64

In [191]:
sprint_2022['Track'].value_counts()

Track
Emilia Romagna    20
Austria           20
Brazil            20
Name: count, dtype: int64

In [192]:
sprint_2023['Track'].value_counts()

Track
Austria          20
Belgium          20
United States    20
Qatar            20
Brazil           20
Azerbaijan       19
Name: count, dtype: int64

In [193]:
sprint_2024['Track'].value_counts()

Track
China            20
Miami            20
Austria          20
United States    20
Brazil           20
Qatar            20
Name: count, dtype: int64

In [198]:
len(kaggle_sprint) 

360

In [200]:
len(sprint_2022) + len(sprint_2023) + len(sprint_2024) + 60 ## 60 is for 2021 which we have to make

359

In [215]:
race = pd.read_csv('filler-datasets/races_filtered.csv')
race[race['year']==2023]

,raceId,year,round,circuitId,name
103,1098,2023,1,3,Bahrain Grand Prix
104,1099,2023,2,77,Saudi Arabian Grand Prix
105,1100,2023,3,1,Australian Grand Prix
106,1101,2023,4,73,Azerbaijan Grand Prix
107,1102,2023,5,79,Miami Grand Prix
108,1104,2023,6,6,Monaco Grand Prix
109,1105,2023,7,4,Spanish Grand Prix
110,1106,2023,8,7,Canadian Grand Prix
111,1107,2023,9,70,Austrian Grand Prix
112,1108,2023,10,9,British Grand Prix


In [217]:
kaggle_sprint_ajer = kaggle_sprint[kaggle_sprint['raceId']==1101]
kaggle_sprint_ajer

,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,fastestLapTime,statusId
120,121,1101,815,9,11,2,1,1,1,8,17,33:17.667,1997667,11,1:43.616,1
121,122,1101,844,6,16,1,2,2,2,7,17,+4.463,2002130,12,1:43.646,1
122,123,1101,830,9,1,3,3,3,3,6,17,+5.065,2002732,10,1:43.723,1
123,124,1101,847,131,63,4,4,4,4,5,17,+3.467,2001134,10,1:44.409,1
124,125,1101,832,6,55,5,5,5,5,4,17,+10.388,2008055,11,1:44.398,1
125,126,1101,4,117,14,8,6,6,6,3,17,+11.613,2009280,12,1:44.544,1
126,127,1101,1,131,44,6,7,7,7,2,17,+4.890,2002557,8,1:44.601,1
127,128,1101,840,117,18,9,8,8,8,1,17,+18.417,2016084,14,1:44.855,1
128,129,1101,848,3,23,7,9,9,9,0,17,+21.757,2019424,10,1:44.915,1
129,130,1101,857,1,81,11,10,10,10,0,17,+22.851,2020518,8,1:44.776,1


In [218]:
# Reassign the dataframe to exclude that specific ID
kaggle_sprint = kaggle_sprint[kaggle_sprint['resultId'] != 140]
len(kaggle_sprint)

359

In [219]:
kaggle_sprint.to_csv('filler-datasets/kaggle_sprint_res.csv', index=False)

In [222]:
sprint_2023[sprint_2023['Position']=='1']

,Track,Position,No,Driver,Team,Starting Grid,Laps,Time/Retired,Points,kaggle_driver_id
0,Azerbaijan,1,11,Sergio Perez,Red Bull Racing Honda RBPT,2,17,33:17.667,8,815
19,Austria,1,1,Max Verstappen,Red Bull Racing Honda RBPT,1,24,30:26.730,8,830
39,Belgium,1,1,Max Verstappen,Red Bull Racing Honda RBPT,1,11,24:58.433,8,830
59,Qatar,1,81,Oscar Piastri,McLaren Mercedes,1,19,35:01.297,8,857
79,United States,1,1,Max Verstappen,Red Bull Racing Honda RBPT,1,19,31:30.849,8,830
99,Brazil,1,1,Max Verstappen,Red Bull Racing Honda RBPT,2,24,30:07.209,8,830


## IT IS WRONG LOGIC AND NO DO NOT GIVE PROPER RESULTS , IT IS FIXED LATER

In [232]:
def assign_sprint_race_ids_simple(sprint_dfs, kaggle_sprint, start_idx=60):
    kaggle_idx = start_idx
    out = []

    for df in sprint_dfs:
        df = df.copy()
        race_ids = []
        i = 0
        n = len(df)

        while i < n:
            race_id = kaggle_sprint.iloc[kaggle_idx]['raceId']

            if race_id == 1101:
                size = 19
            else:
                size = 20

            # 🔴 FIX: don’t overshoot
            size = min(size, n - i)

            race_ids.extend([race_id] * size)

            i += size
            kaggle_idx += 1

        # final safety check
        assert len(race_ids) == len(df)

        df['raceId'] = race_ids
        out.append(df)

    print("✅ Sprint raceId assignment successful (Azerbaijan handled)")
    return out


In [233]:
sprint_2022, sprint_2023, sprint_2024 = assign_sprint_race_ids_simple(
    sprint_dfs=[sprint_2022, sprint_2023, sprint_2024],
    kaggle_sprint=kaggle_sprint,
    start_idx=60
)


✅ Sprint raceId assignment successful (Azerbaijan handled)


In [235]:
for year, df in zip([2022, 2023, 2024],
                    [sprint_2022, sprint_2023, sprint_2024]):
    print(year, df.groupby('raceId').size().value_counts())


2022 60    1
Name: count, dtype: int64
2023 119    1
Name: count, dtype: int64
2024 120    1
Name: count, dtype: int64


In [238]:
kaggle_sprint_2021 = kaggle_sprint.iloc[:60].copy()
sprint_2021 = pd.DataFrame({
    'raceId': kaggle_sprint_2021['raceId'],
    'Position': kaggle_sprint_2021['positionOrder'],
    'No': kaggle_sprint_2021['number'],
    'Starting Grid': kaggle_sprint_2021['grid'],
    'Laps': kaggle_sprint_2021['laps'],
    'Points': kaggle_sprint_2021['points'],
    'Time/Retired': kaggle_sprint_2021['time'],
    'kaggle_driver_id': kaggle_sprint_2021['driverId']
})

In [239]:
sprint_2021 = sprint_2021.merge(
    kaggle_driver[['driverId', 'forename', 'surname']],
    left_on='kaggle_driver_id',
    right_on='driverId',
    how='left'
)

sprint_2021['Driver'] = (
    sprint_2021['forename'] + ' ' + sprint_2021['surname']
)

sprint_2021.drop(columns=['driverId', 'forename', 'surname'], inplace=True)


In [240]:
sprint_2021.head()

,raceId,Position,No,Starting Grid,Laps,Points,Time/Retired,kaggle_driver_id,Driver
0,1061,1,33,2,17,3,25:38.426,830,Max Verstappen
1,1061,2,44,1,17,2,+1.430,1,Lewis Hamilton
2,1061,3,77,3,17,1,+7.502,822,Valtteri Bottas
3,1061,4,16,4,17,0,+11.278,844,Charles Leclerc
4,1061,5,4,6,17,0,+24.111,846,Lando Norris


In [248]:
sprint_2021 = sprint_2021[
    [
        'Position',
        'No',
        'Driver',
        'Starting Grid',
        'Laps',
        'Time/Retired',
        'Points',
        'kaggle_driver_id',
        'raceId'
    ]
].sort_values(['raceId', 'Position']).reset_index(drop=True)


In [249]:
sprint_2021.to_csv('sprint_2021_results.csv', index = False)

In [254]:
race_datasets = {
    2021: sprint_2021,
    2022: sprint_2022,
    2023: sprint_2023,
    2024: sprint_2024,
    2025: sprint_2025,
}
for year, df in race_datasets.items():
    df[['finished', 'dnf', 'laps_down', 'time_gap_sec']] = \
        df['Time/Retired'].apply(parse_time_gap_retired)

    df.drop(columns=['Time/Retired'], inplace=True)

    print(f"✅ {year} processed")


✅ 2021 processed
✅ 2022 processed
✅ 2023 processed
✅ 2024 processed
✅ 2025 processed


In [259]:
def clean_race_columns(df):
    """
    Keeps only the final approved race-result columns.
    Points are explicitly removed.
    """

    cols_to_keep = [
        'Track',
        'Team',
        'raceId',
        'kaggle_driver_id',
        'Position',
        'Starting Grid',
        'finished',
        'dnf',
        'laps_down',
        'time_gap_sec'
    ]

    df.drop(
        columns=[c for c in df.columns if c not in cols_to_keep],
        inplace=True
    )

    return df
race_datasets = {
    2022: sprint_2022,
    2023: sprint_2023,
    2024: sprint_2024,
    2025: sprint_2025
}

for year, df in race_datasets.items():
    race_datasets[year] = clean_race_columns(df)
    print(f"✅ {year} cleaned")


✅ 2022 cleaned
✅ 2023 cleaned
✅ 2024 cleaned
✅ 2025 cleaned


In [262]:
race_datasets = {
    2021: sprint_2021
}

for year, df in race_datasets.items():
    race_datasets[year] = clean_race_columns(df)
    print(f"✅ {year} cleaned")

✅ 2021 cleaned


In [263]:
sprint_2021.to_csv('sprint_2021_result.csv', index=False)
sprint_2022.to_csv('sprint_2022_result.csv', index=False)
sprint_2023.to_csv('sprint_2023_result.csv', index=False)
sprint_2024.to_csv('sprint_2024_result.csv', index=False)
sprint_2025.to_csv('sprint_2025_result.csv', index=False)

## Have to add Constructor and Track to Race Results and Sprint Result 2021 which we will add later

## Now working on qualifying

In [269]:
quali_2025 = pd.read_csv('formula1-dataset-allresults/Formula1_2025Season_QualifyingResults.csv')
quali_2024 = pd.read_csv('formula1-dataset-allresults/Formula1_2024season_qualifyingResults.csv')
quali_2023 = pd.read_csv('formula1-dataset-allresults/Formula1_2023season_qualifyingResults.csv')
quali_2022 = pd.read_csv('formula1-dataset-allresults/Formula1_2022season_qualifyingResults.csv')
sprint_quali_2021 = pd.read_csv('formula1-dataset-allresults/formula1_2021season_sprintQualifyingResults.csv')
sprint_quali_2023 = pd.read_csv('formula1-dataset-allresults/Formula1_2023season_sprintShootoutResults.csv')
sprint_quali_2024 = pd.read_csv('formula1-dataset-allresults/Formula1_2024season_sprintQualifyingResults.csv')
sprint_quali_2021 = pd.read_csv('formula1-dataset-allresults/Formula1_2025Season_SprintQualifyingResults.csv')

In [273]:
quali_2022.head()

,Track,Position,No,Driver,Team,Q1,Q2,Q3,Laps
0,Bahrain,1,16,Charles Leclerc,Ferrari,1:31.471,1:30.932,1:30.558,15
1,Bahrain,2,1,Max Verstappen,Red Bull Racing RBPT,1:31.785,1:30.757,1:30.681,14
2,Bahrain,3,55,Carlos Sainz,Ferrari,1:31.567,1:30.787,1:30.687,15
3,Bahrain,4,11,Sergio Perez,Red Bull Racing RBPT,1:32.311,1:31.008,1:30.921,18
4,Bahrain,5,44,Lewis Hamilton,Mercedes,1:32.285,1:31.048,1:31.238,17


In [268]:
quali_2022.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 440 entries, 0 to 439
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Track     440 non-null    object
 1   Position  440 non-null    object
 2   No        440 non-null    int64 
 3   Driver    440 non-null    object
 4   Team      440 non-null    object
 5   Q1        440 non-null    object
 6   Q2        328 non-null    object
 7   Q3        215 non-null    object
 8   Laps      440 non-null    int64 
dtypes: int64(2), object(7)
memory usage: 31.1+ KB


In [309]:
quali_datasets = {
    'quali_2022': quali_2022,
    'quali_2023': quali_2023,
    'quali_2024': quali_2024,
    'quali_2025': quali_2025,
    'sprint_quali_2021': sprint_quali_2021,
    'sprint_quali_2023': sprint_quali_2023,
    'sprint_quali_2024': sprint_quali_2024,
    'kaggle_quali' : kaggle_quali,
    'kaggle_quali_from_2022' : kaggle_quali_from_2022,
    
}
for name, df in quali_datasets.items():
    print(name, len(df), "mod 20 =", len(df) % 20)


quali_2022 440 mod 20 = 0
quali_2023 440 mod 20 = 0
quali_2024 478 mod 20 = 18
quali_2025 479 mod 20 = 19
sprint_quali_2021 120 mod 20 = 0
sprint_quali_2023 120 mod 20 = 0
sprint_quali_2024 120 mod 20 = 0
kaggle_quali 2556 mod 20 = 16
kaggle_quali_from_2022 1359 mod 20 = 19


In [308]:
def check_block_sizes(df):
    blocks = []
    start = 0
    tracks = df['Track'].values

    for i in range(1, len(df)):
        if tracks[i] != tracks[i - 1]:
            blocks.append(i - start)
            start = i
    blocks.append(len(df) - start)

    return blocks
for name, df in quali_datasets.items():
    blocks = check_block_sizes(df)
    bad = [b for b in blocks if b != 20]

    if bad:
        print(name, "❌ non-20 blocks:", bad)
    else:
        print(name, "✅ all blocks = 20")


quali_2022 ✅ all blocks = 20
quali_2023 ✅ all blocks = 20
quali_2024 ❌ non-20 blocks: [19, 19]
quali_2025 ❌ non-20 blocks: [19]
sprint_quali_2021 ✅ all blocks = 20
sprint_quali_2023 ✅ all blocks = 20
sprint_quali_2024 ✅ all blocks = 20


In [287]:
def find_bad_blocks(df):
    bad_blocks = []
    start = 0
    tracks = df['Track'].values

    for i in range(1, len(df)):
        if tracks[i] != tracks[i - 1]:
            size = i - start
            if size != 20:
                bad_blocks.append((start, i - 1, size, tracks[start]))
            start = i

    # last block
    size = len(df) - start
    if size != 20:
        bad_blocks.append((start, len(df) - 1, size, tracks[start]))

    return bad_blocks


In [288]:
find_bad_blocks(quali_2024)

[(40, 58, 19, 'Australia'), (279, 297, 19, 'Netherlands')]

In [289]:
find_bad_blocks(quali_2025)


[(400, 418, 19, 'Brazil')]

In [294]:
## checked the real data, it also had 19 drivers only

In [333]:
kaggle_quali = pd.read_csv('filler-datasets/qualifying.csv')
kaggle_quali = kaggle_quali[kaggle_quali['raceId']>1009]
kaggle_quali_from_2022 = kaggle_quali[kaggle_quali['raceId']>1073]
kaggle_quali[kaggle_quali['raceId']==1135]

,qualifyId,raceId,driverId,constructorId,number,position,q1,q2,q3
10294,10352,1135,846,1,4,1,1:11.377,1:10.496,1:09.673
10295,10353,1135,830,9,1,2,1:11.393,1:10.811,1:10.029
10296,10354,1135,857,1,81,3,1:11.541,1:10.505,1:10.172
10297,10355,1135,847,131,63,4,1:11.049,1:10.552,1:10.244
10298,10356,1135,815,9,11,5,1:11.006,1:10.678,1:10.416
10299,10357,1135,844,6,16,6,1:11.370,1:10.689,1:10.582
10300,10358,1135,4,117,14,7,1:11.493,1:10.845,1:10.633
10301,10359,1135,840,117,18,8,1:11.518,1:10.661,1:10.857
10302,10360,1135,842,214,10,9,1:11.718,1:10.815,1:10.977
10303,10361,1135,832,6,55,10,1:11.327,1:10.914,NaN


In [312]:
def find_bad_blocks_kaggle(df):
    bad_blocks = []
    start = 0
    tracks = df['raceId'].values

    for i in range(1, len(df)):
        if tracks[i] != tracks[i - 1]:
            size = i - start
            if size != 20:
                bad_blocks.append((start, i - 1, size, tracks[start]))
            start = i

    # last block
    size = len(df) - start
    if size != 20:
        bad_blocks.append((start, len(df) - 1, size, tracks[start]))

    return bad_blocks


In [313]:
find_bad_blocks_kaggle(kaggle_quali_from_2022)

[(920, 938, 19, np.int64(1123))]

In [331]:
# make it explicit
kaggle_quali_from_2022 = kaggle_quali_from_2022.copy()

# ensure position is string
kaggle_quali_from_2022.loc[:, 'position'] = (
    kaggle_quali_from_2022['position'].astype(str)
)

idx = 10301
race_id = kaggle_quali_from_2022.loc[idx, 'raceId']

# get race block
race_block_idx = kaggle_quali_from_2022.index[
    kaggle_quali_from_2022['raceId'] == race_id
]

# 🔑 store original position BEFORE DQ
orig_pos = kaggle_quali_from_2022.loc[idx, 'position']
assert orig_pos.isdigit(), "Position already modified earlier"

dq_pos = int(orig_pos)

# shift positions after DQ
mask = (
    kaggle_quali_from_2022.index.isin(race_block_idx) &
    kaggle_quali_from_2022['position'].str.isdigit() &
    (kaggle_quali_from_2022['position'].astype(int) > dq_pos)
)

kaggle_quali_from_2022.loc[mask, 'position'] = (
    kaggle_quali_from_2022.loc[mask, 'position'].astype(int) - 1
).astype(str)

# mark DQ + remove times
kaggle_quali_from_2022.loc[idx, ['Q1', 'Q2', 'Q3']] = np.nan
kaggle_quali_from_2022.loc[idx, 'position'] = 'DQ'


C:\Users\ASUS\AppData\Local\Temp\ipykernel_19564\1925859515.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['1' '2' '3' ... '18' '19' '20']' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  kaggle_quali_from_2022.loc[:, 'position'] = (


In [340]:
quali_2023.head()

,Track,Position,No,Driver,Team,Q1,Q2,Q3,Laps,raceId
0,Bahrain,1,1,Max Verstappen,Red Bull Racing Honda RBPT,1:31.295,1:30.503,1:29.708,15,1074
1,Bahrain,2,11,Sergio Perez,Red Bull Racing Honda RBPT,1:31.479,1:30.746,1:29.846,15,1074
2,Bahrain,3,16,Charles Leclerc,Ferrari,1:31.094,1:30.282,1:30.000,17,1074
3,Bahrain,4,55,Carlos Sainz,Ferrari,1:30.993,1:30.515,1:30.154,18,1074
4,Bahrain,5,14,Fernando Alonso,Aston Martin Aramco Mercedes,1:31.158,1:30.645,1:30.336,15,1074


In [7]:
def quali_time_to_sec(val):
    if pd.isna(val):
        return np.nan

    val = str(val).strip()
    parts = val.split(':')

    try:
        if len(parts) == 2:
            # mm:ss.xxx
            m, s = parts
            return int(m) * 60 + float(s)
        elif len(parts) == 3:
            # hh:mm:ss.xxx (rare but exists)
            h, m, s = parts
            return int(h) * 3600 + int(m) * 60 + float(s)
    except:
        return np.nan

    return np.nan


In [334]:
kaggle_quali_from_2022.to_csv('kaggle_quali_from_2022.csv',index=False)

In [335]:
# unique raceIds in order (this is important)
kaggle_race_ids_from_2022 = (
    kaggle_quali_from_2022[['raceId']]
    .drop_duplicates()
    .reset_index(drop=True)['raceId']
    .tolist()
)


In [336]:
def assign_race_id_by_track(df, race_ids):
    df = df.copy()

    race_id_col = []
    race_ptr = 0

    tracks = df['Track'].values
    n = len(df)

    start = 0
    for i in range(1, n):
        if tracks[i] != tracks[i - 1]:
            # assign raceId to previous block
            race_id_col.extend([race_ids[race_ptr]] * (i - start))
            race_ptr += 1
            start = i

    # last block
    race_id_col.extend([race_ids[race_ptr]] * (n - start))
    race_ptr += 1

    # safety
    assert len(race_id_col) == len(df), "raceId length mismatch"

    df['raceId'] = race_id_col
    return df
# race qualifying
quali_2022 = assign_race_id_by_track(quali_2022, kaggle_race_ids_from_2022)
quali_2023 = assign_race_id_by_track(quali_2023, kaggle_race_ids_from_2022)
quali_2024 = assign_race_id_by_track(quali_2024, kaggle_race_ids_from_2022)
quali_2025 = assign_race_id_by_track(quali_2025, kaggle_race_ids_from_2022)


In [337]:
for name, df in {
    'quali_2022': quali_2022,
    'quali_2023': quali_2023,
    'quali_2024': quali_2024,
    'quali_2025': quali_2025,
}.items():
    print(name, df.groupby('raceId').size().min(), df.groupby('raceId').size().max())


quali_2022 20 20
quali_2023 20 20
quali_2024 19 20
quali_2025 19 20


In [341]:
def normalize_name(s):
    return (
        s.str.lower()
         .str.strip()
         .str.normalize('NFKD')
         .str.encode('ascii', errors='ignore')
         .str.decode('utf-8')
    )
quali_2024['driver_norm'] = normalize_name(quali_2024['Driver'])
kaggle_driver['driver_norm'] = (
    kaggle_driver['forename'] + ' ' + kaggle_driver['surname']
)

kaggle_driver['driver_norm'] = normalize_name(kaggle_driver['driver_norm'])
quali_2024 = quali_2024.merge(
    kaggle_driver[['driverId', 'driver_norm']],
    on='driver_norm',
    how='left'
)

In [12]:
def normalize_name_series(s):
    return (
        s.str.lower()
         .str.strip()
         .str.normalize('NFKD')
         .str.encode('ascii', errors='ignore')
         .str.decode('utf-8')
    )

def add_kaggle_driver_id(df, kaggle_driver):
    df = df.copy()

    df['driver_norm'] = normalize_name_series(df['Driver'])

    kd = kaggle_driver.copy()
    kd['driver_norm'] = normalize_name_series(
        kd['forename'] + ' ' + kd['surname']
    )

    df = df.merge(
        kd[['driverId', 'driver_norm']],
        on='driver_norm',
        how='left'
    )

    df.rename(columns={'driverId': 'kaggle_driver_id'}, inplace=True)
    df.drop(columns=['driver_norm'], inplace=True)

    return df


In [16]:
quali_2023 = add_kaggle_driver_id(quali_2023, kaggle_driver)
quali_2025 = add_kaggle_driver_id(quali_2025, kaggle_driver)
sprint_quali_2021 = add_kaggle_driver_id(sprint_quali_2021, kaggle_driver)
sprint_quali_2023 = add_kaggle_driver_id(sprint_quali_2023, kaggle_driver)
sprint_quali_2024 = add_kaggle_driver_id(sprint_quali_2024, kaggle_driver)

In [345]:
for name, df in {
    'quali_2023': quali_2023,
    'quali_2025': quali_2025,
    'sprint_quali_2021': sprint_quali_2021,
    'sprint_quali_2023': sprint_quali_2023,
    'sprint_quali_2024': sprint_quali_2024,
}.items():
    print(name, df['kaggle_driver_id'].isna().sum())


quali_2023 0
quali_2025 0
sprint_quali_2021 0
sprint_quali_2023 0
sprint_quali_2024 0


In [342]:
quali_2024.head()

,Track,Position,No,Driver,Team,Q1,Q2,Q3,Laps,raceId,kaggle_driver_id
0,Bahrain,1,1,Max Verstappen,Red Bull Racing Honda RBPT,1:30.031,1:29.374,1:29.179,17,1074,830
1,Bahrain,2,16,Charles Leclerc,Ferrari,1:30.243,1:29.165,1:29.407,19,1074,844
2,Bahrain,3,63,George Russell,Mercedes,1:30.350,1:29.922,1:29.485,17,1074,847
3,Bahrain,4,55,Carlos Sainz,Ferrari,1:29.909,1:29.573,1:29.507,18,1074,832
4,Bahrain,5,11,Sergio Perez,Red Bull Racing Honda RBPT,1:30.221,1:29.932,1:29.537,17,1074,815


In [8]:
def process_quali(df):
    df = df.copy()

    # participation flags
    df['participated_q2'] = df['Q2'].notna().astype(int)
    df['participated_q3'] = df['Q3'].notna().astype(int)

    # parse times
    df['q1_time_sec'] = df['Q1'].apply(quali_time_to_sec)
    df['q2_time_sec'] = df['Q2'].apply(quali_time_to_sec)
    df['q3_time_sec'] = df['Q3'].apply(quali_time_to_sec)

    # keep final columns
    df = df[
        [
            'raceId',
            'Track',
            'Team',
            'kaggle_driver_id',
            'Position',
            'participated_q2',
            'participated_q3',
            'q1_time_sec',
            'q2_time_sec',
            'q3_time_sec',
        ]
    ]

    return df


In [34]:
quali_2024 = process_quali(quali_2024)
quali_2025 = process_quali(quali_2025)

In [32]:
quali_2024 = add_kaggle_driver_id(quali_2024, kaggle_driver)

In [ ]:
quali_2024 = add_kaggle_driver_id(quali_2023, kaggle_driver)
quali_2024 = process_quali(quali_2022)

In [350]:
quali_2022.to_csv("race_quali_2022.csv", index=False)
quali_2023.to_csv("race_quali_2023.csv", index=False)
quali_2024.to_csv("race_quali_2024.csv", index=False)
quali_2025.to_csv("race_quali_2025.csv", index=False)
sprint_quali_2021.to_csv("sprint_quali_2021.csv", index=False)
sprint_quali_2023.to_csv("sprint_quali_2023.csv", index=False)
sprint_quali_2024.to_csv("sprint_quali_2024.csv", index=False)

In [3]:
quali_2025 = pd.read_csv('race_quali_2022.csv')
quali_2024 = pd.read_csv('race_quali_2022.csv')
quali_2023 = pd.read_csv('race_quali_2022.csv')
quali_2022 = pd.read_csv('race_quali_2022.csv')
sprint_quali_2021 = pd.read_csv('sprint_quali_2021.csv')
sprint_quali_2023 = pd.read_csv('sprint_quali_2021.csv')
sprint_quali_2024 = pd.read_csv('sprint_quali_2021.csv')
sprint_quali_2021 = pd.read_csv('sprint_quali_2021.csv')

In [42]:
sprint_quali_2024.head()

,Track,Position,No,Driver,Team,Q1,Q2,Q3,Laps,kaggle_driver_id,kaggle_driver_id
0,China,1,44,Lewis Hamilton,Ferrari,1:31.212,1:31.484,1:30.849,15,1,1
1,China,2,1,Max Verstappen,Red Bull Racing Honda RBPT,1:31.916,1:31.521,1:30.867,12,830,830
2,China,3,81,Oscar Piastri,McLaren Mercedes,1:31.723,1:31.362,1:30.929,13,857,857
3,China,4,16,Charles Leclerc,Ferrari,1:31.518,1:31.561,1:31.057,15,844,844
4,China,5,63,George Russell,Mercedes,1:31.952,1:31.346,1:31.169,18,847,847


In [14]:
kaggle_driver = pd.read_csv('filler-datasets/drivers_filtered.csv')